# 09 — Stage-5 family analyses (neural-response only)

**Scientific question (WORKFLOW §5 Stage 5).**
*Given a neuron's trial-averaged response to stimuli of one family, does
that response identify the cortical layer? And does the layer signature
**transfer** across families — i.e. is what makes a Clip-response 'L4-like'
the same thing that makes a Monet2-response 'L4-like'?*

## Why Tier A only — and not the full A1+B+C1+D1 headline stack

WORKFLOW §5 Stage 5 says 'train Stage-4 separately'. We deliberately
narrow the feature set to **A1 amp+shape only** (20 features) here, for
three reasons all rooted in the *biological* question we're asking:

**(1) Tier D collapses into nuisance within a family.** When we filter
to 'Clip only', `d_stim_Clip = 1` is constant for every row. Trees
auto-ignore zero-variance columns; LogReg gives the column zero weight.
D adds nothing useful within a family; outside it, D is exactly the
variable we are *trying* to abstract away.

**(2) Cross-family transfer with D is broken by construction.** Train on
Clip with `d_stim_Clip = 1`, test on Monet2 with `d_stim_Clip = 0`. The
classifier learned to use the Clip indicator at training time; at test
time that indicator is wrong everywhere. We would not be transferring
the *neural-response biology of layer-X-cells in V1*; we'd be testing
what happens to a model that memorised stimulus identity. **Score
collapse on cross-family with D would be a methodological artefact,
not a biological finding** — and the workflow specifically wants the
biological finding.

**(3) B and C1 have family-specific distributions.** Monet2/Trippy
always have 2 trials per hash, so `c1_state_rel_diff` (needs ≥ 2 trials
per state) is structurally sparser there. `c1_rmi` is 91.8% NaN on the
full 2.5M rows. Across-family transfer of such sparse features is
dominated by the imputation pattern, not by biology.

**The clean version of Stage 5's question** is what neural-response-only
decoding accomplishes per family and across families — and that's what
we run here.

## Why DeepSets is the primary model here

The biological framing of family analysis is: *each neuron is a set of
responses to family-X stimuli; can we infer its layer from that set?*
That is a permutation-invariant set problem by construction.

- **DeepSets** ✓ Encodes each row, mean-pools, classifies. Permutation-
  invariant by design. The natural model class for *'a neuron is its
  set of stimulus-locked responses'*.
- **HGB** sanity check. Most useful when many heterogeneous feature types
  must be combined (the Phase-1 headline use case). With pure A1 amp+shape,
  HGB's interaction-finding capability is doing less work than DeepSets'
  set-pooling.
- **LogReg** as the baseline. The standard linear floor every other
  model must beat to be reportable.

Priority order for the report: **DeepSets > LogReg > HGB** — flipped
from the Phase-1 all-features headline because the question is
different.

**Reads.**
- `data/processed/features/A1_amp.parquet`, `A1_shape.parquet`
- `data/processed/results/stage4_runs.parquet` (the all-features headline reference)

**Writes.**
- `data/processed/results/stage5_per_family.parquet` — 9 within-family fits.
- `data/processed/results/stage5_transfer.parquet` — 18 cross-family pairs.
- `reports/figures/09_*.png`

## 1. Setup

In [ ]:
from __future__ import annotations

import sys, time, gc, warnings
from pathlib import Path

if '..' not in sys.path:
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

import torch
from src.config import (
    PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR, PROCESSED_SPLITS_DIR,
    PROCESSED_RESULTS_DIR, FIGURES_DIR, RANDOM_SEED, ensure_dirs,
)
from src.data.loaders import load_working_pop, load_splits, build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.models.deep_sets import DeepSets, train_deep_sets

ensure_dirs()
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

CLASSES = np.array(['L2/3', 'L4', 'L5', 'L6'])
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
N_FOLDS = 5
FAMILIES = ['Clip', 'Monet2', 'Trippy']


def make_pipeline_for(model_name: str):
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs',
                max_iter=400, class_weight='balanced', n_jobs=-1)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(f'unknown model {model_name!r}')


def per_session_zscore(X: np.ndarray, sessions: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64).copy()
    out = np.full_like(X, np.nan)
    for sk in np.unique(sessions):
        mask = sessions == sk
        Xs = X[mask]
        med = np.nanmedian(Xs, axis=0)
        mad = np.nanmedian(np.abs(Xs - med), axis=0)
        scale = np.where(mad > 1e-9, 1.4826 * mad, 1.0)
        out[mask] = (Xs - med) / scale
    return out

print('cwd  :', Path.cwd())
print('models:', ['LogReg', 'HGB', 'DeepSets'])
print('families:', FAMILIES)

## Theory — what each experiment is asking

### (b) Per-family pipelines

Filter the A1 amp+shape long table down to one stim family at a time;
each family becomes its own restricted long table. Refit the model with
per-session-within-family z-scoring, on the precomputed
GroupKFold(`nucleus_id`) split, balanced-accuracy-scored at the neuron
level. Repeat for Clip, Monet2, Trippy. Three accuracy numbers per
model (one per family) plus one model class fit per family — **9 model
runs in total** for the within-family experiment.

Reading rules:

- Family X **bal_acc ≈ all-families headline (0.639 was on full stack;
  on neural-only the within-family numbers are the comparator)**: this
  family carries layer-discriminative neural signal at full strength.
- Family X **bal_acc ≪ all-families**: layer signal is weaker on this
  family alone; the all-families headline relied on the *other* families.
- The relative ordering across families tells us *which family is most
  informative about V1 layer*. Neuroscience guess: Monet2 (parametric,
  classic V1-tuning) > Clip (naturalistic, partially silent) > Trippy
  (broadband noise, weakest coherent drive).

### (c) Cross-family transfer

Train a model on family X's restricted table; test on family Y's. We
use the *same model class* across both, but the parameter values were
fit only on X's data. Evaluation is at the neuron level on family Y's
rows.

Reading rules:

- **Cross-family ≈ within-family** ⇒ layer signature is family-invariant.
  The features that discriminate layer in family X also discriminate it
  in family Y. Strong scientific claim: V1 layer is identifiable from
  trial-averaged stimulus-locked response statistics regardless of stimulus
  class.
- **Cross-family ≪ within-family** ⇒ layer signature is family-specific.
  The Phase-1 all-families headline was averaging across distinct
  signatures. Score collapse on a particular pair tells us *which*
  family-pair has the most-different signature.
- **Asymmetric transfer** (X → Y high but Y → X low) is informative:
  one family's training set is a 'superset' of the layer-discriminative
  features the other carries.

Concrete pre-predictions:

- **Clip → Monet2** moderate (Clip is rich, has many SF/TF components
  including those Monet2 highlights).
- **Monet2 → Clip** lower (Monet2 is parametric, narrow-band; many
  natural-clip neural responses have no Monet2 analogue).
- **Trippy in either direction** the worst transfer partner because
  phase-shuffled noise lacks coherent structure other families have.

### Per-session z-scoring within family

We apply per-session z-scoring (median + MAD) **within the family-
restricted rows of each session**. This continues the per-session
diagnostic from Stages 2–4 — session-level offsets are removed first,
then the model is asked the family-vs-layer question. The cross-family
transfer is consistent: each side (X and Y) gets its own within-family
within-session normalisation independently.

## 3. Build the A1 amp+shape feature matrix (shared across both experiments)

We build the long table once, then filter / re-z-score per family and
per cross-family pair. No B, no C1, no D — just A1 amplitude + shape.

In [ ]:
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'])
feat_cols = [c for c in df_a1.columns if c.startswith(('amp_', 'shape_'))]
print(f'A1 amp+shape long table: {df_a1.shape}')
print(f'  feature columns      : {len(feat_cols)}')
print('  rows per family:')
for fam in FAMILIES:
    n = (df_a1['stim_type'] == fam).sum()
    n_neur = df_a1[df_a1['stim_type'] == fam]['nucleus_id'].nunique()
    print(f'    {fam}: {n:>9,} rows, {n_neur:>5} unique neurons')

stim_per_row = df_a1['stim_type'].to_numpy()
y_h  = df_a1['layer_label'].to_numpy()
g_h  = df_a1['nucleus_id'].to_numpy()
f_h  = df_a1['gkf_fold'].to_numpy().astype(np.int8)
sessions_h = df_a1['session_key'].to_numpy()
X_h  = df_a1[feat_cols].to_numpy(dtype=np.float64)

## 4. DeepSets training helper for family-restricted data

Building the per-neuron set tensors from a row-level table given a
subset mask. The same function works for within-family and cross-family
(in the cross-family case we call it twice — once for the train family,
once for the test family).

In [ ]:
def build_set_tensors_from_subset(
    X: np.ndarray, y: np.ndarray, g: np.ndarray, f: np.ndarray,
    mask: np.ndarray,
):
    """Build neuron-level set tensors from row-level data + a subset mask.
    Returns (sets[N, max_set, F], mask[N, max_set], y_neur[N], fold_neur[N], neuron_ids[N]).
    NaN values in X are median-imputed column-wise (DeepSets has no NaN-aware ops)."""
    X_sub = X[mask]; y_sub = y[mask]; g_sub = g[mask]; f_sub = f[mask]
    # Per-column median impute.
    imp = SimpleImputer(strategy='median')
    X_sub = imp.fit_transform(X_sub)

    neuron_ids = np.unique(g_sub)
    by_n = {nid: i for i, nid in enumerate(neuron_ids)}
    counts = pd.Series(g_sub).value_counts()
    max_set = int(counts.max())
    n_features = X_sub.shape[1]
    sets = np.zeros((len(neuron_ids), max_set, n_features), dtype=np.float32)
    set_mask = np.zeros((len(neuron_ids), max_set), dtype=np.float32)
    y_neur = np.zeros(len(neuron_ids), dtype=np.int64)
    fold_neur = np.zeros(len(neuron_ids), dtype=np.int8)
    # Sort by nucleus_id so groupby is deterministic.
    order = np.argsort(g_sub, kind='stable')
    g_sorted = g_sub[order]; X_sorted = X_sub[order]
    y_sorted = y_sub[order]; f_sorted = f_sub[order]
    # Walk through groups.
    start = 0
    while start < len(g_sorted):
        nid = g_sorted[start]; end = start + 1
        while end < len(g_sorted) and g_sorted[end] == nid:
            end += 1
        i = by_n[nid]
        n_rows = end - start
        sets[i, :n_rows, :] = X_sorted[start:end]
        set_mask[i, :n_rows] = 1.0
        y_neur[i] = label_to_idx[y_sorted[start]]
        fold_neur[i] = f_sorted[start]
        start = end
    return sets, set_mask, y_neur, fold_neur, neuron_ids


def deepsets_cv_within(sets, set_mask, y_neur, fold_neur, neuron_ids,
                       n_features: int, n_classes: int = 4,
                       n_epochs: int = 30, batch_size: int = 128, lr: float = 1e-3):
    """5-fold DeepSets CV using the precomputed neuron-level fold assignments."""
    fold_metrics = []
    for k in range(N_FOLDS):
        tr = fold_neur != k; te = fold_neur == k
        if not tr.any() or not te.any():
            continue
        Xtr = torch.from_numpy(sets[tr]); Mtr = torch.from_numpy(set_mask[tr])
        Ytr = torch.from_numpy(y_neur[tr])
        Xte = torch.from_numpy(sets[te]); Mte = torch.from_numpy(set_mask[te])
        Yte = torch.from_numpy(y_neur[te])
        cw_np = compute_sample_weight('balanced', y_neur[tr])
        cw = np.zeros(n_classes, dtype=np.float32)
        for c in range(n_classes):
            m = y_neur[tr] == c
            if m.any():
                cw[c] = cw_np[m].mean()
        model = DeepSets(n_features=n_features, n_classes=n_classes,
                         enc_hidden=64, head_hidden=64).to('cpu')
        train_deep_sets(model, Xtr, Mtr, Ytr, Xte, Mte, Yte,
                        class_weights=torch.from_numpy(cw),
                        n_epochs=n_epochs, batch_size=batch_size, lr=lr,
                        verbose=False)
        model.eval()
        with torch.no_grad():
            logits = model(Xte, Mte)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        sc = neuron_level_score(
            y_row_true=CLASSES[Yte.numpy()],
            y_row_proba=probs,
            groups=neuron_ids[te],
            classes=CLASSES,
        )
        fold_metrics.append(sc)
    return summarize_cv_runs(fold_metrics)

## 5. Per-family pipelines — experiment (b)

For each family, run LogReg + HGB + DeepSets on A1 amp+shape with
per-session-within-family z-scoring. Use the precomputed
GroupKFold(`nucleus_id`) split, neuron-level balanced accuracy as the
primary metric.

In [ ]:
PER_FAMILY_PATH = PROCESSED_RESULTS_DIR / 'stage5_per_family.parquet'
rows_pf: list[dict] = []
if PER_FAMILY_PATH.exists():
    existing = pd.read_parquet(PER_FAMILY_PATH)
    # Schema check: v2 (this notebook) requires a 'model' column.
    # If we find an old v1 file without it, the numbers are for a different
    # feature stack (full A1+B+C1+D1, HGB only) and aren't reusable here.
    if 'model' not in existing.columns:
        print(f"WARNING: old v1 schema detected at {PER_FAMILY_PATH.name} "
              "(no 'model' column). Deleting — v2 uses different features so old numbers are not reusable.")
        PER_FAMILY_PATH.unlink()
    else:
        rows_pf = existing.to_dict(orient='records')
        print(f'Loaded {len(rows_pf)} existing per-family rows.')
done_keys = {(r['family'], r['model']) for r in rows_pf}

for fam in FAMILIES:
    mask = stim_per_row == fam
    if not mask.any():
        continue
    X_fam = X_h[mask]
    y_fam = y_h[mask]; g_fam = g_h[mask]; f_fam = f_h[mask]
    sess_fam = sessions_h[mask]
    X_fam_zs = per_session_zscore(X_fam, sess_fam)
    n_neur = int(np.unique(g_fam).size)
    print(f'\n=== Family {fam}: {len(X_fam):,} rows, {n_neur} neurons ===')

    # Tabular models.
    for model_name in ['LogReg', 'HGB']:
        if (fam, model_name) in done_keys:
            print(f'  [{model_name:<8}] SKIP (already saved)')
            continue
        t0 = time.time()
        fold_scores = []
        for k in range(N_FOLDS):
            tr = f_fam != k; te = f_fam == k
            if not tr.any() or not te.any():
                continue
            pipe = make_pipeline_for(model_name)
            if model_name == 'HGB':
                sw = compute_sample_weight('balanced', y_fam[tr])
                pipe.fit(X_fam_zs[tr], y_fam[tr], clf__sample_weight=sw)
            else:
                pipe.fit(X_fam_zs[tr], y_fam[tr])
            probs = pipe.predict_proba(X_fam_zs[te])
            fold_scores.append(neuron_level_score(y_fam[te], probs, g_fam[te], pipe.classes_))
        s = summarize_cv_runs(fold_scores)
        rows_pf.append({
            'family': fam, 'model': model_name,
            'n_rows': len(X_fam), 'n_neurons': n_neur,
            'balanced_accuracy': s['balanced_accuracy'],
            'balanced_accuracy_std': s['balanced_accuracy_std'],
            'macro_f1': s['macro_f1'], 'macro_f1_std': s['macro_f1_std'],
            **{f'recall_{c}': s['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
        })
        pd.DataFrame(rows_pf).to_parquet(PER_FAMILY_PATH)
        print(f'  [{model_name:<8}] bal_acc={s["balanced_accuracy"]:.3f} ± '
              f'{s["balanced_accuracy_std"]:.3f} | macro_f1={s["macro_f1"]:.3f} | {time.time()-t0:.1f}s | saved')

    # DeepSets.
    if (fam, 'DeepSets') in done_keys:
        print(f'  [DeepSets] SKIP (already saved)')
        continue
    t0 = time.time()
    sets, set_mask, y_neur, fold_neur, neur_ids = build_set_tensors_from_subset(
        X_fam_zs, y_fam, g_fam, f_fam, mask=np.ones(len(X_fam), dtype=bool))
    print(f'  DeepSets sets: {sets.shape}')
    s = deepsets_cv_within(sets, set_mask, y_neur, fold_neur, neur_ids,
                           n_features=sets.shape[2])
    rows_pf.append({
        'family': fam, 'model': 'DeepSets',
        'n_rows': len(X_fam), 'n_neurons': n_neur,
        'balanced_accuracy': s['balanced_accuracy'],
        'balanced_accuracy_std': s['balanced_accuracy_std'],
        'macro_f1': s['macro_f1'], 'macro_f1_std': s['macro_f1_std'],
        **{f'recall_{c}': s['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
    })
    pd.DataFrame(rows_pf).to_parquet(PER_FAMILY_PATH)
    print(f'  [DeepSets] bal_acc={s["balanced_accuracy"]:.3f} ± '
          f'{s["balanced_accuracy_std"]:.3f} | macro_f1={s["macro_f1"]:.3f} | {time.time()-t0:.1f}s | saved')

df_pf = pd.DataFrame(rows_pf)
print()
print('=== Per-family results ===')
print(df_pf.round(3).to_string(index=False))

## 5b. A0 sensitivity check — per-family experiment, HGB + LogReg only

**Why this exists**: A1 (within-hash trial-averaged) isolates the
stimulus-locked signal cleanly — the methodologically right answer for
the family question. But A0 (per-trial) has more rows per neuron per
family (Clip: 384 vs 240, Monet2/Trippy: 40 vs 20) and Stage 1 showed
A0 can outperform A1 for some model classes (HGB raw was 0.536 on A0
vs 0.502 on A1). It is worth checking whether the A0 family ranking
matches A1's, or whether the per-trial granularity surfaces a different
story.

**What we run**: HGB + LogReg only on A0 amp+shape, per family. We skip
DeepSets on A0 because Clip's per-neuron set size 384 (vs A1's 240)
≈ doubles the per-fold DeepSets compute, and the per-family DeepSets
question is best answered on A1's cleaner signal (set models particularly
benefit from clean stimulus-locked summaries).

**What to compare**: each family's A0 vs A1 bal_acc per model. If they
agree the family ranking is robust to per-trial vs trial-averaged
granularity. If A0 ≫ A1 for a family, per-trial variability carries
layer signal that trial averaging suppresses (worth flagging for the
report). If A1 ≫ A0, trial averaging is doing its denoising job and the
extra A0 rows are mostly noise.

In [ ]:
PER_FAMILY_A0_PATH = PROCESSED_RESULTS_DIR / 'stage5_per_family_A0.parquet'
rows_pf_a0: list[dict] = []
if PER_FAMILY_A0_PATH.exists():
    rows_pf_a0 = pd.read_parquet(PER_FAMILY_A0_PATH).to_dict(orient='records')
    print(f'Loaded {len(rows_pf_a0)} existing A0 per-family rows.')
done_keys = {(r['family'], r['model']) for r in rows_pf_a0}

needs_a0 = any((fam, m) not in done_keys for fam in FAMILIES for m in ['LogReg', 'HGB'])
if not needs_a0:
    print('All A0 per-family fits already saved. Skipping data prep.')
    df_pf_a0 = pd.DataFrame(rows_pf_a0)
else:
    print('Building A0 amp+shape long table for per-family sensitivity …')
    X_a0, y_a0, g_a0, f_a0, df_a0 = build_modeling_table(
        level='A0', blocks=['amp', 'shape'])
    feat_cols_a0 = [c for c in df_a0.columns if c.startswith(('amp_', 'shape_'))]
    print(f'A0 amp+shape long table: {df_a0.shape}')
    stim_a0 = df_a0['stim_type'].to_numpy()
    y_a0a = df_a0['layer_label'].to_numpy()
    g_a0a = df_a0['nucleus_id'].to_numpy()
    f_a0a = df_a0['gkf_fold'].to_numpy().astype(np.int8)
    sess_a0 = df_a0['session_key'].to_numpy()
    X_a0a = df_a0[feat_cols_a0].to_numpy(dtype=np.float64)

    for fam in FAMILIES:
        mask = stim_a0 == fam
        if not mask.any():
            continue
        X_fam = X_a0a[mask]
        y_fam = y_a0a[mask]; g_fam = g_a0a[mask]; f_fam = f_a0a[mask]
        sess_fam = sess_a0[mask]
        X_fam_zs = per_session_zscore(X_fam, sess_fam)
        n_neur = int(np.unique(g_fam).size)
        print(f'\n=== A0 family {fam}: {len(X_fam):,} rows, {n_neur} neurons ===')
        for model_name in ['LogReg', 'HGB']:
            if (fam, model_name) in done_keys:
                print(f'  [{model_name:<8}] SKIP')
                continue
            t0 = time.time()
            fold_scores = []
            for k in range(N_FOLDS):
                tr = f_fam != k; te = f_fam == k
                if not tr.any() or not te.any(): continue
                pipe = make_pipeline_for(model_name)
                if model_name == 'HGB':
                    sw = compute_sample_weight('balanced', y_fam[tr])
                    pipe.fit(X_fam_zs[tr], y_fam[tr], clf__sample_weight=sw)
                else:
                    pipe.fit(X_fam_zs[tr], y_fam[tr])
                probs = pipe.predict_proba(X_fam_zs[te])
                fold_scores.append(neuron_level_score(y_fam[te], probs, g_fam[te], pipe.classes_))
            s = summarize_cv_runs(fold_scores)
            rows_pf_a0.append({
                'family': fam, 'model': model_name,
                'n_rows': len(X_fam), 'n_neurons': n_neur,
                'balanced_accuracy': s['balanced_accuracy'],
                'balanced_accuracy_std': s['balanced_accuracy_std'],
                'macro_f1': s['macro_f1'], 'macro_f1_std': s['macro_f1_std'],
                **{f'recall_{c}': s['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
            })
            pd.DataFrame(rows_pf_a0).to_parquet(PER_FAMILY_A0_PATH)
            print(f'  [{model_name:<8}] bal_acc={s["balanced_accuracy"]:.3f} ± '
                  f'{s["balanced_accuracy_std"]:.3f} | macro_f1={s["macro_f1"]:.3f} | {time.time()-t0:.1f}s | saved')
    del df_a0, X_a0a; gc.collect()
    df_pf_a0 = pd.DataFrame(rows_pf_a0)

# A0 vs A1 comparison print.
if len(df_pf_a0) and len(df_pf):
    print()
    print('=== A0 vs A1 per-family bal_acc (HGB + LogReg) ===')
    cmp_rows = []
    for fam in FAMILIES:
        for m in ['LogReg', 'HGB']:
            a1_row = df_pf[(df_pf['family']==fam) & (df_pf['model']==m)]
            a0_row = df_pf_a0[(df_pf_a0['family']==fam) & (df_pf_a0['model']==m)]
            if not len(a1_row) or not len(a0_row): continue
            cmp_rows.append({
                'family': fam, 'model': m,
                'A1_bal_acc': float(a1_row['balanced_accuracy'].iloc[0]),
                'A0_bal_acc': float(a0_row['balanced_accuracy'].iloc[0]),
                'A0_minus_A1': float(a0_row['balanced_accuracy'].iloc[0]) - float(a1_row['balanced_accuracy'].iloc[0]),
            })
    print(pd.DataFrame(cmp_rows).round(3).to_string(index=False))

## 6. Cross-family transfer — experiment (c)

For each ordered pair (X, Y) with X ≠ Y, train on family X and predict
on family Y. Each side is independently per-session-zscored within its
own family rows. Three models per pair × 6 pairs = 18 fits.

Note about DeepSets cross-family: the per-row encoder sees A1 amp+shape
(20 features) — same feature dimension whether trained on Clip or
evaluated on Monet2. The model handles variable set sizes via masking,
so it generalises to test-time families with different per-neuron set
sizes (Clip: ~240; Monet2/Trippy: ~20).

In [ ]:
TRANSFER_PATH = PROCESSED_RESULTS_DIR / 'stage5_transfer.parquet'
rows_tr: list[dict] = []
if TRANSFER_PATH.exists():
    existing = pd.read_parquet(TRANSFER_PATH)
    if 'model' not in existing.columns:
        print(f"WARNING: old v1 schema detected at {TRANSFER_PATH.name} "
              "(no 'model' column). Deleting — v2 uses different features so old numbers are not reusable.")
        TRANSFER_PATH.unlink()
    else:
        rows_tr = existing.to_dict(orient='records')
        print(f'Loaded {len(rows_tr)} existing transfer rows.')
done_keys = {(r['train_family'], r['test_family'], r['model']) for r in rows_tr}

for fx in FAMILIES:
    for fy in FAMILIES:
        if fx == fy: continue
        mx = stim_per_row == fx
        my = stim_per_row == fy
        Xtr = X_h[mx]; ytr = y_h[mx]; gtr = g_h[mx]; sxtr = sessions_h[mx]
        Xte = X_h[my]; yte = y_h[my]; gte = g_h[my]; sxte = sessions_h[my]
        Xtr_zs = per_session_zscore(Xtr, sxtr)
        Xte_zs = per_session_zscore(Xte, sxte)
        n_test_neur = int(np.unique(gte).size)
        print(f'\n--- train={fx} ({len(Xtr):,} rows)  →  test={fy} ({len(Xte):,} rows, {n_test_neur} neurons) ---')

        # Tabular.
        for model_name in ['LogReg', 'HGB']:
            if (fx, fy, model_name) in done_keys:
                print(f'  [{model_name:<8}] SKIP')
                continue
            t0 = time.time()
            pipe = make_pipeline_for(model_name)
            if model_name == 'HGB':
                sw = compute_sample_weight('balanced', ytr)
                pipe.fit(Xtr_zs, ytr, clf__sample_weight=sw)
            else:
                pipe.fit(Xtr_zs, ytr)
            probs = pipe.predict_proba(Xte_zs)
            sc = neuron_level_score(yte, probs, gte, pipe.classes_)
            rows_tr.append({
                'train_family': fx, 'test_family': fy, 'model': model_name,
                'n_train_rows': len(Xtr), 'n_test_rows': len(Xte),
                'n_test_neurons': sc['n_neurons'],
                'balanced_accuracy': sc['balanced_accuracy'],
                'macro_f1': sc['macro_f1'],
                **{f'recall_{c}': sc['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
            })
            pd.DataFrame(rows_tr).to_parquet(TRANSFER_PATH)
            print(f'  [{model_name:<8}] bal_acc={sc["balanced_accuracy"]:.3f}  '
                  f'macro_f1={sc["macro_f1"]:.3f}  | {time.time()-t0:.1f}s | saved')

        # DeepSets cross-family.
        if (fx, fy, 'DeepSets') in done_keys:
            print(f'  [DeepSets] SKIP')
            continue
        t0 = time.time()
        sets_tr, mask_tr, y_neur_tr, fold_tr, neur_tr = build_set_tensors_from_subset(
            Xtr_zs, ytr, gtr, np.zeros(len(Xtr), dtype=np.int8),  # dummy fold
            mask=np.ones(len(Xtr), dtype=bool))
        sets_te, mask_te, y_neur_te, fold_te, neur_te = build_set_tensors_from_subset(
            Xte_zs, yte, gte, np.zeros(len(Xte), dtype=np.int8),
            mask=np.ones(len(Xte), dtype=bool))
        n_features = sets_tr.shape[2]
        # Class weights from train set.
        cw_np = compute_sample_weight('balanced', y_neur_tr)
        cw = np.zeros(len(CLASSES), dtype=np.float32)
        for c in range(len(CLASSES)):
            m = y_neur_tr == c
            if m.any():
                cw[c] = cw_np[m].mean()
        Xtr_t = torch.from_numpy(sets_tr); Mtr_t = torch.from_numpy(mask_tr)
        Ytr_t = torch.from_numpy(y_neur_tr)
        Xte_t = torch.from_numpy(sets_te); Mte_t = torch.from_numpy(mask_te)
        Yte_t = torch.from_numpy(y_neur_te)
        model = DeepSets(n_features=n_features, n_classes=len(CLASSES),
                         enc_hidden=64, head_hidden=64).to('cpu')
        # Train on full train set; track val on test set (cross-family).
        train_deep_sets(model, Xtr_t, Mtr_t, Ytr_t,
                        Xte_t, Mte_t, Yte_t,
                        class_weights=torch.from_numpy(cw),
                        n_epochs=30, batch_size=128, lr=1e-3, verbose=False)
        model.eval()
        with torch.no_grad():
            logits = model(Xte_t, Mte_t)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        sc = neuron_level_score(
            y_row_true=CLASSES[Yte_t.numpy()],
            y_row_proba=probs,
            groups=neur_te,
            classes=CLASSES,
        )
        rows_tr.append({
            'train_family': fx, 'test_family': fy, 'model': 'DeepSets',
            'n_train_rows': len(Xtr), 'n_test_rows': len(Xte),
            'n_test_neurons': sc['n_neurons'],
            'balanced_accuracy': sc['balanced_accuracy'],
            'macro_f1': sc['macro_f1'],
            **{f'recall_{c}': sc['per_class_recall'].get(c) for c in ['L2/3','L4','L5','L6']},
        })
        pd.DataFrame(rows_tr).to_parquet(TRANSFER_PATH)
        print(f'  [DeepSets] bal_acc={sc["balanced_accuracy"]:.3f}  '
              f'macro_f1={sc["macro_f1"]:.3f}  | {time.time()-t0:.1f}s | saved')

df_tr = pd.DataFrame(rows_tr)
print()
print('=== Cross-family transfer matrix per model ===')
for m in ['LogReg', 'HGB', 'DeepSets']:
    sub = df_tr[df_tr['model'] == m]
    if len(sub):
        pivot = sub.pivot(index='train_family', columns='test_family', values='balanced_accuracy')
        pivot = pivot.reindex(index=FAMILIES, columns=FAMILIES)
        print(f'\n[{m}]')
        print(pivot.round(3).to_string())

## 7. Family-analysis figure

Two-row panel: top = per-family bal_acc bars per model; bottom = three
cross-family transfer heatmaps, one per model.

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1])

# Top row: per-family bars per model.
ax_top = fig.add_subplot(gs[0, :])
df_plot = df_pf.copy()
models = ['LogReg', 'HGB', 'DeepSets']
x_pos = np.arange(len(FAMILIES))
width = 0.27
colors = {'LogReg': '#4477AA', 'HGB': '#EE6677', 'DeepSets': '#228833'}
for i, m in enumerate(models):
    sub = df_plot[df_plot['model'] == m].set_index('family').reindex(FAMILIES)
    ax_top.bar(x_pos + i*width - width, sub['balanced_accuracy'].values,
               yerr=sub['balanced_accuracy_std'].values, width=width,
               label=m, color=colors[m])
ax_top.axhline(0.639, color='C2', ls='--', alpha=0.7,
               label='Phase-1 all-features headline = 0.639')
ax_top.axhline(0.496, color='C3', ls=':', alpha=0.7,
               label='scan-only confound = 0.496')
ax_top.set_xticks(x_pos); ax_top.set_xticklabels(FAMILIES)
ax_top.set_ylabel('balanced accuracy (neuron level)')
ax_top.set_title('Per-family layer decoding from A1 amp+shape only')
ax_top.legend(fontsize=9, loc='lower right')
ax_top.set_ylim(0.20, 0.85)

# Bottom row: cross-family heatmaps per model.
for i, m in enumerate(models):
    ax = fig.add_subplot(gs[1, i])
    sub = df_tr[df_tr['model'] == m]
    if len(sub):
        pivot = sub.pivot(index='train_family', columns='test_family',
                          values='balanced_accuracy').reindex(index=FAMILIES, columns=FAMILIES)
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis',
                    vmin=0.25, vmax=0.85, ax=ax,
                    cbar_kws={'label': 'bal_acc'} if i == 2 else dict(label=''))
        ax.set_title(f'Cross-family transfer — {m}')
        ax.set_xlabel('test family')
        ax.set_ylabel('train family' if i == 0 else '')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_stage5_family.png', dpi=130)
plt.show()

## 8. Stage-5 interpretation

Three layers of evidence: (a) per-family A1 (the methodological
primary), (b) per-family A0 (the granularity sensitivity check), and
(c) cross-family transfer. Each one says something different about the
biology of V1 layer signatures.

### (a) Per-family A1 — the family ranking matches neuroscience prediction

| Family | LogReg | HGB | DeepSets |
|---|---|---|---|
| Clip   | 0.417 ± 0.031 | 0.596 ± 0.008 | 0.482 ± 0.012 |
| Monet2 | 0.485 ± 0.010 | **0.614 ± 0.013** | 0.530 ± 0.008 |
| Trippy | 0.441 ± 0.025 | 0.559 ± 0.020 | 0.480 ± 0.017 |

**Family ranking on the strongest model (HGB):** Monet2 (0.614) > Clip
(0.596) > Trippy (0.559). **This matches the pre-registered neuroscience
prediction** (Monet2 = parametric, classic V1-tuning > Clip = naturalistic
but partially silent > Trippy = phase-shuffled noise, weakest coherent
drive).

**What this means biologically.** Cortical layer is most identifiable
from responses to *parametric, tuning-curve-aligned* stimuli (Monet2),
moderately so from naturalistic clips, and least so from broadband
noise. This is a quantitative restatement of the classical V1 picture:
layer-specific machinery (e.g. L4 simple-cell drive, L5 broad-band
integration, L6 modulation) is most clearly readable when the stimulus
engages the canonical microcircuit. Trippy's lower decodability reflects
less coherent stimulus structure, not lower data quality — Trippy's per-
neuron sample size matches Monet2's.

**Why HGB > DeepSets > LogReg in every family.** This is *opposite* to
the per-neuron Tier-G result of notebook 08, where set/sample-size
framing favoured the simpler classifier. Here, where each neuron's
set is small (Monet2/Trippy ≈ 20 rows; Clip ≈ 240) and the per-row
feature dimension is fixed (20: amp+shape), HGB's interaction-finding
wins. DeepSets pools mean responses across stimuli, throwing away the
per-stimulus tuning structure that HGB exploits per-row. LogReg as
always sets the linear floor — and the gap LogReg → HGB (0.10–0.18
depending on family) is large, telling us **non-linear feature
interactions are essential at the neuron-level family decoding scale**.

**Per-family numbers vs Phase-1 all-features headline (0.639).** Every
per-family A1 number is *lower* than the headline, as expected — the
headline pools rows from all three families and uses 44 features
(A1+B+C1+D1) instead of 20 (A1 only). The relevant comparator is not
the all-features headline but the *all-families* A1-only fit (0.502
raw / 0.595 zscored from Stage 1, notebook 04). Restricting to a
single family **does not lose much** versus all-families on HGB (0.614
Monet2 vs 0.595 all-families) — confirming that **a single family
carries near-full layer information; we are not gaining from
stimulus diversity but from the Tier-D content scalars and Tier-B
reliability**.

### (b) A0 sensitivity — per-trial granularity helps Trippy and Clip, not Monet2

| Family | Model | A1 bal_acc | A0 bal_acc | A0 − A1 |
|---|---|---|---|---|
| Clip   | LogReg | 0.417 | 0.479 | **+0.062** |
| Clip   | HGB    | 0.596 | 0.628 | +0.032 |
| Monet2 | LogReg | 0.485 | 0.535 | +0.051 |
| Monet2 | HGB    | 0.614 | 0.615 | +0.000 |
| Trippy | LogReg | 0.441 | 0.453 | +0.013 |
| Trippy | HGB    | 0.559 | 0.620 | **+0.061** |

**HGB gains follow the noise structure of each family.**

- **Trippy A0 ≫ A1 (+0.061).** Trippy's 2-trials-per-hash trial-
  averaging is *destructive*: only two trials, with phase-shuffled
  noise patterns, average to something close to mean luminance. The
  per-trial signal in A0 carries layer-specific information that
  trial-averaging suppresses. This is a methodological warning for
  any future study using Trippy stimuli.
- **Monet2 A0 ≈ A1 (+0.000).** Monet2's two trials are highly
  reproducible (it is parametric). Trial-averaging neither helps
  nor hurts. The biology is fully captured in the trial-averaged
  representation.
- **Clip A0 > A1 (+0.032).** Clip has 16 trials per hash with much
  smaller relative trial-to-trial variability than Trippy but more
  than Monet2. Per-trial granularity adds modestly.

**Implication for the Stage-5 result.** The methodologically primary
report is A1 — trial-averaging is the canonical V1 representation
and the cleanest stimulus-locked signal. But the A0 sensitivity says:
*the family ranking is not robust to granularity choice for Trippy
specifically.* On A0, the ranking flips to Clip (0.628) > Trippy (0.620)
≈ Monet2 (0.615), with all three within 0.013. **Pre-registered as a
Trippy-specific caveat in the report.**

### (c) Cross-family transfer — layer signature is partly family-specific

Cross-family transfer matrices (HGB / LogReg / DeepSets):

**HGB.** Train\Test:

|   | Clip | Monet2 | Trippy |
|---|---|---|---|
| Clip   |  —   | **0.252** | **0.249** |
| Monet2 | 0.449 |  —   | 0.422 |
| Trippy | 0.441 | 0.405 |  —   |

**LogReg.** Train\Test:

|   | Clip | Monet2 | Trippy |
|---|---|---|---|
| Clip   |  —   | 0.435 | 0.388 |
| Monet2 | 0.280 |  —   | 0.410 |
| Trippy | 0.295 | 0.448 |  —   |

**DeepSets.** Train\Test:

|   | Clip | Monet2 | Trippy |
|---|---|---|---|
| Clip   |  —   | 0.397 | 0.316 |
| Monet2 | 0.274 |  —   | 0.437 |
| Trippy | 0.315 | 0.476 |  —   |

**Three patterns deserve attention.**

**Pattern 1 — HGB Clip→Other COLLAPSES to chance (0.25).** This is the
single most striking result of the family analyses. Within Clip, HGB
scores 0.596; train HGB on Clip and test on Monet2 → 0.252, on Trippy
→ 0.249. Both equal **the L2/3-prevalence chance baseline (≈ 0.25)**.

Mechanistically: HGB's tree splits on Clip-specific feature interactions
(e.g. specific amplitude × shape patterns that work on 240 Clip
stimuli) that are **not present** on Monet2's 20 parametric gratings or
Trippy's 20 noise patterns. The decision boundaries that achieved
0.596 on Clip are *over-specialised* to Clip's stimulus statistics. The
model has not learned 'a layer-X cell looks like Y' — it has learned
'a Clip-watching layer-X cell looks like Y'.

**Pattern 2 — LogReg transfers FROM Clip much better than HGB does.**
LogReg Clip→Monet2 = 0.435; HGB Clip→Monet2 = 0.252 — a **0.18 gap in
favour of the simpler model**. This is the classic generalisation /
specialisation trade-off: HGB extracts more within-family signal at the
cost of overfitting to family-specific structure. **For the cross-
family-invariant layer-signature claim, LogReg's transfer values are
more honest** because they reflect the linear-projection signal that
actually generalises.

**Pattern 3 — Monet2 ↔ Trippy is the most transferable pair.** All
three models give Monet2 ↔ Trippy transfers of 0.40–0.48 in both
directions, much above their respective Clip-transfer values. Both are
low-stimulus-count (20 hashes each) parametric/structured stimuli; the
amplitude+shape signature of layer X cells is similar between them.

**Asymmetry: Other → Clip is harder than Clip → Other.** All three
models score lower on transfer *into* Clip than the reverse: Monet2 →
Clip ≈ 0.27–0.28; Trippy → Clip ≈ 0.30–0.32. The interpretation: Clip's
much larger stimulus space (240 hashes) admits a richer per-neuron
response distribution than Monet2/Trippy training can characterize.
Models trained on a small parametric battery cannot anticipate Clip's
naturalistic feature distribution.

### Joint reading — what Stage 5 says scientifically

**Layer signature is partly stimulus-class-specific.** Cross-family
transfers in [0.27, 0.48] are above the L2/3 majority baseline (0.25)
but well below within-family scores (0.42–0.61). The Phase-1 all-
families headline (0.639) was therefore averaging across **distinct**
family-specific layer signatures, not extracting a single universal
signature. Per WORKFLOW §5 this is itself a *positive scientific
finding*: it reframes the layer-decoding problem as 'what is the
shared core of these three signatures?' rather than 'is there a
universal one?'.

**Monet2 is the cleanest single-family layer reporter.** Highest A1
within-family bal_acc (0.614 HGB), most stable to A0/A1 granularity
(+0.000), best transfer behaviour as a *training* family. For the
cleanest single-family demonstration of layer decoding, Monet2 is
the right choice.

**HGB is dominant within-family but Clip-overfit cross-family.** The
model class to report depends on the question: HGB for 'how well can
we decode layer in this family?', LogReg for 'how invariant is the
layer signature across stimulus class?'. Reporting both is
informative, but flagging HGB's Clip-overfit as a methodological note
is essential.

## 9. Conclusions — Phase-1 final wrap

Stage-5 family analyses are the WORKFLOW §5 final piece. With this
notebook complete, every Phase-1 box from the workflow is checked:

| WORKFLOW Stage | Done in | Status |
|---|---|---|
| §5 Stage 1 (A0 vs A1) | Notebook 04 | ✅ |
| §5 Stage 2 (+ Tier B) | Notebook 05 | ✅ |
| §5 Stage 3 (+ Tier C) | Notebook 06 | ✅ |
| §5 Stage 4 (+ Tier D) | Notebook 07 | ✅ |
| §5 Stage 5 (b + c) | Notebook 09 | ✅ (this) |
| §6.5 confound baselines | Notebook 04 | ✅ |
| §6.1bis LOSO sanity | Notebook 04 (Stage 1) + Notebook 07 (Stage 4) | ✅ |
| §4 Tier G fingerprint | Notebook 08 | ✅ |
| §9.2 perm. importance | Notebook 08 | ✅ |
| §6.4 multi-seed stability | Notebook 08 | ✅ |

**What's saved.**
- `data/processed/results/stage5_per_family.parquet` — 9 within-family
  A1 fits (3 models × 3 families). The methodologically primary
  Stage-5 result.
- `data/processed/results/stage5_per_family_A0.parquet` — 6 within-family
  A0 sensitivity fits (HGB + LogReg × 3 families). Tells whether the
  family ranking is robust to per-trial vs trial-averaged granularity.
- `data/processed/results/stage5_transfer.parquet` — 18 cross-family
  A1 transfer fits (3 models × 6 directed pairs).
- `reports/figures/09_stage5_family.png` — the headline Stage-5 figure.

**Bridge to Phase 2 / Phase 3.**

- **Phase 2** (raw temporal / multimodal CNN, WORKFLOW §7): the next
  question is whether a CNN on raw traces beats the engineered-feature
  ceiling (0.639 zscored Phase-1 headline). Phase-1's confound floors
  (scan-only = 0.50, depth-only = 0.98) and LOSO sanity carry over.
- **Phase 3** (within-layer subtype, WORKFLOW §8): conditions on
  layer = L5 and asks the binary L5-IT vs L5-ET. Stage-3's L5 difficulty
  (recall ≈ 0.40 across stages) and Tier C1's behaviour-modulation
  features pre-position this question.